# Terminal as an offline Linux desktop app (Electron)

This notebook is an implementation guide for packaging this repository's Angular front end and ASP.NET Core API as an installable Linux desktop app. The goal is a single-user application that starts a local API automatically, stores data in the user's home directory, and works without internet for features implemented locally.

**Scope:** Electron, Linux x64 first, existing Angular UI and ASP.NET API. End users should not need Node.js, npm, Docker, or the .NET runtime.

**Offline means:** the UI, API, SQLite data, tasks, and results work locally. Commands that open ChatGPT, LinkedIn, Indeed, or other remote services still require internet.

Work through each phase in order, keep the browser workflow working, and make small commits.

## 1. Current project facts

- `src/terminal-front` is Angular and already has a production build. Its production API URL is `/terminal-api`, for the hosted site; desktop needs runtime configuration.
- `src/TerminalApi` is an ASP.NET Core 10 service using SQLite and EF Core migrations.
- The API requires `Auth__Username`, `Auth__Password`, and `Jwt__Key`. `init.sh` is for development and creates a temporary key.
- `TERMINAL_SQLITE_PATH` can point SQLite at an absolute path. Tasks and results currently resolve relative to the API executable directory unless configured, so desktop must move those to user data too.
- Terminal commands run inside `TerminalRunner` sessions. Check commands for assumptions about the repository working directory, shell, and installed programs.

This is primarily a packaging and integration effort. Risks are process lifecycle, local configuration, Linux paths, and exposing the API address to Angular.

## 2. Target architecture

1. Electron main process chooses an available loopback port and starts the API bound only to `127.0.0.1`.
2. It passes the API URL, database path, task/result paths, credentials, and signing key through a controlled child-process environment.
3. It waits for API readiness, then serves the Angular production files from a small local HTTP server on loopback.
4. Electron opens that local page. Runtime configuration gives Angular the API URL. Keep Node and filesystem access out of the renderer.
5. On exit, Electron stops the UI server and API child; it waits briefly and only force-stops the child it created.

Serving Angular over local HTTP avoids common `file://` origin/CORS problems. Bind both listeners to `127.0.0.1`, never `0.0.0.0`. Select ports dynamically or handle conflicts with a clear error. Configure CORS for the actual UI origin, or build a same-origin proxy and correctly support SignalR WebSockets.

Use Electron with `contextIsolation: true`, `nodeIntegration: false`, and `sandbox: true`. Restrict navigation to the app origin. Open approved external links in the system browser. Never load remote scripts into the desktop app.

## 3. Linux build prerequisites

The build machine needs Git, Node.js LTS with npm, .NET 10 SDK, and native tools required by Electron packaging. End users of the package should not need those runtimes. Check the current Electron/`electron-builder` Linux requirements for the chosen versions.

Start with a named Linux x64 distribution/version. AppImage is convenient as a single download; `.deb` is convenient for Debian/Ubuntu. Test on the distribution you claim to support; one successful build does not establish compatibility with every Linux distribution.

At the time this guide was prepared, this environment has .NET but `npm` is not on PATH. Install Node.js/npm before building the Angular or Electron projects.

## 4. Create an Electron workspace

Create `src/terminal-desktop/` as a separate project. Add pinned Electron and `electron-builder` development dependencies and commit the lockfile. A suggested layout:

```text
src/terminal-desktop/
  package.json
  electron/main.cjs
  electron/preload.cjs
  electron/local-web-server.cjs
  icons/
```

Keep Electron runtime code outside Angular. Do not expose generic filesystem or shell methods through preload. Prefer a narrow API or no preload API. External links should be opened with Electron's system-browser API after validating their scheme and destination.

Choose the final reverse-domain application identifier, app name, and Linux icon. Keep development startup scripts separate from release packaging scripts.

## 5. Configure the API URL at runtime

`TerminalService` currently captures `environment.apiBaseUrl`. Desktop ports should be selected at startup, so introduce typed runtime configuration loaded before the API service is used. For example, Electron's local server can return `desktop-config.json` with `{ "apiBaseUrl": "http://127.0.0.1:<port>" }`. Preserve current environment defaults for browser development and hosted production.

Implementation steps:

- Load config through an Angular application initializer or a bootstrap step that fetches JSON before creating services.
- Do not put passwords or JWT signing keys in this JSON.
- If using separate UI/API ports, allow only the actual local UI origin in API CORS and check both HTTP and SignalR/WebSocket flows.
- Alternatively, proxy `/api` and `/ws` through the local UI server to the API and keep same-origin browser requests. Verify WebSocket upgrades and bind the proxy to loopback. Pick one approach and document it.

Keep `environment.ts` and `environment.prod.ts` useful for the existing web workflow.

## 6. Publish the API self-contained for Linux

Publish a Linux x64 build so users do not install .NET. From the repository root, start with:

```bash
dotnet publish src/TerminalApi/TerminalApi.csproj \
  --configuration Release \
  --runtime linux-x64 \
  --self-contained true \
  -p:PublishSingleFile=true \
  -p:IncludeNativeLibrariesForSelfExtract=true \
  --output src/terminal-desktop/artifacts/api/linux-x64
```

Confirm the output contains runtime dependencies, SQLite native assets, and required content/configuration. If single-file publishing causes issues, publish the full self-contained output directory instead. Ensure the executable bit is preserved. Ignore generated artifacts in Git.

Electron should spawn the API executable directly, not through a shell. Capture stdout/stderr into bounded logs and show a useful error if startup fails. Set `TERMINAL_SQLITE_PATH` to an absolute user-data database path. Also set ASP.NET environment keys `FilePaths__Tasks` and `FilePaths__Results` to absolute paths under Electron's `app.getPath('userData')`, creating directories first. Verify migrations run against that path.

Use app data, never the install directory or `app.asar`, for mutable files: database, tasks, results, configuration, and logs. Updates may replace install files, and system package locations may not be writable.

## 7. Credentials, startup, and shutdown

Plan first launch because the API will not start without username, password, and JWT key:

1. Ask the user to create local credentials, or generate a random password and reveal it once. Never ship a shared default password.
2. Store configuration in the app-data directory with owner-only permissions (`0700` directory, `0600` file). Prefer the OS credential store for the password if its added dependency is acceptable. Never log secrets or pass them as command-line arguments.
3. Generate a cryptographically random JWT key and persist it locally so restarts do not invalidate tokens. Keep this key private.
4. Initially retain the current sign-in screen, using only the local account for the local API.

Choose an available port, spawn the service with `--urls http://127.0.0.1:<port>`, and poll the existing anonymous `GET /api/version` endpoint with a timeout. A dedicated health endpoint can be added later. On timeout, show the log location. Handle app close and crashes so no orphan API process is left behind.

Loopback binding limits network exposure, but other processes as the same OS user can still connect. Authentication is useful defense in depth, not a substitute for loopback-only binding and protected files.

## 8. Build Angular assets and package Electron

In `src/terminal-front`, run `npm ci` and `npm run build`. Confirm the browser output path (commonly `dist/terminal-front/browser` for this Angular builder). Serve the packaged files with the local static server; implement Angular history fallback, correct MIME types, and path-traversal protection.

The desktop release script should:

1. Build Angular production assets.
2. Publish the self-contained API for the target architecture.
3. Package only Electron code, Angular output, and API publish output.
4. Emit an AppImage; add `.deb` after that path is working.

Declare scripts and artifact paths in `src/terminal-desktop/package.json`. Ignore generated output. Exclude repository source, credentials, databases, user data, and build caches. Pin versions and state the supported architecture. For releases, provide checksums. Linux code signing can wait for the initial personal release.

## 9. Manual release checks on a clean Linux account

Record distribution, version, architecture, package type, and each outcome. These are manual release checks, not automated tests.

- Install and launch without Node.js/npm/.NET installed; the user should not need to use a terminal.
- Disconnect from the internet. Sign in, initialize, and use core local features. Verify remote-service commands fail gracefully or open the browser without crashing the app.
- Create/update data, quit, relaunch, and confirm DB, tasks, and results persist in the user's app-data directory.
- Confirm listeners bind only to `127.0.0.1`. Close the app and confirm its API child exits.
- Occupy the preferred port or launch a second copy; confirm port recovery or a clear error.
- Verify sign-in after restart, and ensure logs contain no secrets. Confirm another OS user cannot read the config/data directory.
- Check app routes, keyboard input, session events/SignalR, resizing, and external links.
- Upgrade to a newer package; confirm user data persists. Back up and restore the database.
- Test the actual AppImage/`.deb` on each distribution you claim to support.

For the offline claim, inspect network requests during core flows as well as disconnecting Wi-Fi. Confirm the core UI is served from packaged local assets and does not depend on a cached remote resource.

## 10. Sequence and completion criteria

**A — process proof:** minimal Electron window, self-contained API launch, loopback binding, clean shutdown, persistent SQLite.

**B — UI integration:** runtime API URL, local static serving, CORS or same-origin proxy, sign-in and terminal session input/output.

**C — first-run hardening:** local account, persistent JWT key, private file permissions, port conflicts, timeouts, and logs.

**D — distribution:** AppImage, then `.deb` if needed; clean-install and upgrade checks.

A prototype may take a few days for someone familiar with Angular, ASP.NET, and Electron. A reliable personal Linux release is more likely one to three weeks. The biggest uncertainty is how terminal commands behave when launched from a packaged child process outside the repository working directory. Resolve this in phase A.

### Completion definition

A normal Linux user can install and launch the app without developer runtimes, use core local features with the network disconnected, and retain data through app restarts and upgrades. Features that call remote services are clearly outside the offline guarantee.